# Azure Auto ML for Computer Vision
This notebook is a quick tutorial/guide on how to use Azure AutoML for computer vision. You can also checkout this [Microsoft tutorial](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-auto-train-image-models?view=azureml-api-2&tabs=python).

*Note: In order to use AutoML for computer vision you need to have a **GPU compute cluster**.*

# Notebook Setup

Set project paths and load workspace MLClient.

In [1]:
import os
# Here we set the working directory to the project root to ensure imports work correctly
from pathlib import Path
target = "dp100-learn"
p = Path.cwd()
print(f"Starting working directory: {p}")
while p.name != target and p.parent != p:
    p = p.parent
# Set the path to your project root manually if the above code does not work
p = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn"
# p = "C:/Users/dmika/DEV/Projects-local/dp100-learn"
os.chdir(p)
print("Changed working directory to:", p)
from utils.azureml_utils import *

# Get Azure ML Client based on your environment. Learn more in the tutorials/azureml-first-notebook.ipynb.
ml_client = get_azureml_client()

Starting working directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code
Changed working directory to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn
Added to sys.path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn


Found the config file in: /config.json


Added to sys.path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn


# Prepare Data

Firt we need data. We'll work on [Intel Image Classification Dataset](https://www.kaggle.com/datasets/puneet6060/intel-image-classification) that can be found on kaggle.

In [3]:
from pathlib import Path
# Setup paths
dataset_dir = Path(os.path.join(project_dir, "data/intel-image-classification"))
subset_dir = dataset_dir / "subset"
subset_dir.mkdir(exist_ok=True)
subset_data_dir = subset_dir / "data"
subset_data_dir.mkdir(exist_ok=True)
annotations_dir = subset_dir / "annotations"
annotations_dir.mkdir(exist_ok=True)

## Download and extract the data locally

You can manually download the dataset from kaggle or you can use kaggle API to download it using the code snippet below. To do that you need to authenticate with Kaggle API by following instructions here: https://www.kaggle.com/docs/api.

In [ ]:
import kaggle

# Authenticate kaggle api
kaggle.api.authenticate()
# Download the Intel Image Classification dataset and unzip it
kaggle.api.dataset_download_files('puneet6060/intel-image-classification', path=dataset_dir, unzip=True)

Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification


## Rearange the data into correct folder structure
The train, test and pred folders are dobule nested. To create a simpler structure we will rearange them into single folders.

In [4]:
# Flatten directory structure
import shutil

base = dataset_dir
for folder in ["seg_train", "seg_test", "seg_pred"]:
    inner = base / folder / folder
    if inner.exists():
        for item in inner.iterdir():
            shutil.move(str(item), str(base / folder))
        shutil.rmtree(inner)

## Create a subset for faster experimentation (optional)

We will create a subset of the data for experimantation. It will save resources and costs on Azure.
We will take 200 images from each class ($6\cdot200=1200$).

In [ ]:
# Take a subset of the training data for quicker experiments
import random

for class_dir in (dataset_dir / "seg_train").iterdir():
    if class_dir.is_dir():
        files = list(class_dir.glob("*.jpg"))
        sample_files = random.sample(files, min(200, len(files)))
        target_dir = subset_data_dir / class_dir.name
        target_dir.mkdir(exist_ok=True)
        for f in sample_files:
            shutil.copy(f, target_dir / f.name)
print(f"Subset created at: {subset_data_dir}")

Subset created at: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn/data/intel-image-classification/subset/data


## Create Azure Data Asset

The data needs to be in a specific format to be able to use it for Azure AutoML. Follow the next steps to set up a data asset for AutoML. For more information refer to this [guide](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-prepare-datasets-for-automl-images?view=azureml-api-2&tabs=python).

### Upload the images

First you need to upload the images to azure. We can do that by create URI_FOLDER data asset.

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

my_data = Data(
    path=str(subset_data_dir),
    datastore="dmdp100",
    type=AssetTypes.URI_FOLDER,
    description="Subset of Intel Image Classification dataset for AutoML image training",
    name="intel-image-subset-folder",
)
uploaded_data = ml_client.data.create_or_update(my_data)

### Annotations in JSONL file

Then we need to prepare the annotations - image path and its label in a JSONL file. The path will point to the images we uploaded earlier.

In [ ]:
import json

img_data_asset = ml_client.data.get(name="intel-image-subset-folder", version="2")
base_uri = img_data_asset.path

annotations_dir = subset_dir / "annotations"
annotations_dir.mkdir(exist_ok=True)
jsonl_path = annotations_dir / "train_annotations.jsonl"
subset_data_dir = subset_dir / "data"
records = []

for class_dir in subset_data_dir.iterdir():
    if class_dir.is_dir():
        label = class_dir.name
        for img in class_dir.glob("*.jpg"):
            rel = img.relative_to(subset_data_dir)
            records.append({"image_url": f"{base_uri}{rel.as_posix()}", "label": label})

with open(jsonl_path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

print("✅ JSONL created:", jsonl_path)

✅ JSONL created: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn/data/intel-image-classification/subset/train_annotations.jsonl


### Create MLTable file

Then we need to create MLTable file to point to the annotations file. Remember that the path to the image has to be of `stream_info` type.

In [ ]:
%%writefile data/intel-image-classification/subset/annotations/MLTable

paths:
  - file: ./train_annotations.jsonl
transformations:
  - read_json_lines:
        encoding: utf8
        invalid_lines: error
        include_path_column: false
  - convert_column_types:
      - columns: image_url
        column_type: stream_info   

Overwriting data/intel-image-classification/subset/MLTable


### Create a data asset

Finally we can create an MLTable data asset which can be used by AutoML.

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

mltable_asset = Data(
    name="intel-image-subset-mltable",
    path=str(annotations_dir),
    type=AssetTypes.MLTABLE,
    datastore="dmdp100",
    description="Intel Image Classification subset prepared for AutoML"
)
registered_mltable = ml_client.data.create_or_update(mltable_asset)
print("✅ Registered MLTable asset:", registered_mltable.id)


Uploading subset (18.36 MBs): 100%|██████████| 18364632/18364632 [00:10<00:00, 1692236.44it/s]




✅ Registered MLTable asset: /subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/data/intel-image-subset-mltable/versions/3


# Running Auto ML for Image Classification

## Load Inputs


Now we can load an MLTable asset we've created earlier.

In [55]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input

# creates a dataset based on the files in the local data folder
my_training_data_input = Input(type=AssetTypes.MLTABLE, path="azureml:intel-image-subset-mltable:4")

# Training MLTable defined locally, with local data to be uploaded
# my_training_data_input = Input(type=AssetTypes.MLTABLE, path=training_mltable_path)
# Validation MLTable defined locally, with local data to be uploaded
# my_validation_data_input = Input(type=AssetTypes.MLTABLE, path=validation_mltable_path)
# WITH REMOTE PATH: If available already in the cloud/workspace-blob-store
# my_training_data_input = Input(type=AssetTypes.MLTABLE, path="azureml://datastores/workspaceblobstore/paths/vision-classification/train")
# my_validation_data_input = Input(type=AssetTypes.MLTABLE, path="azureml://datastores/workspaceblobstore/paths/vision-classification/valid")

## Configure classification job

In [56]:
from azure.ai.ml import automl

image_classification_job = automl.image_classification(
    compute="dmdp100-gpu-cluster",
    experiment_name="dmdp100-automl-img-classification",
    display_name="intel-imgs-subset-classification-automl",
    training_data=my_training_data_input,
    target_column_name="label"
)

In [57]:
# Set limits
image_classification_job.set_limits(
    timeout_minutes=120,
    max_trials=10,
    max_concurrent_trials=3,
)

## Run classification job

In [58]:
# Submit the AutoML job
returned_job = ml_client.jobs.create_or_update(
    image_classification_job
)  

# Other

Uncategorized code snippets

In [37]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

my_data = Data(
    path=str(subset_dir / "data"),
    datastore="dmdp100",
    type=AssetTypes.URI_FOLDER,
    description="Subset of Intel Image Classification dataset for AutoML image training",
    name="intel-image-subset-folder",
)
uploaded_data = ml_client.data.create_or_update(my_data)

In [ ]:
uri_folder_data_asset  = ml_client.data.get(name="intel-image-subset-folder", version="2")

In [44]:
import json

img_data_asset = ml_client.data.get(name="intel-image-subset-folder", version="2")
base_uri = img_data_asset.path

annotations_dir = subset_dir / "annotations"
annotations_dir.mkdir(exist_ok=True)
jsonl_path = annotations_dir / "train_annotations.jsonl"
subset_data_dir = subset_dir / "data"
records = []

for class_dir in subset_data_dir.iterdir():
    if class_dir.is_dir():
        label = class_dir.name
        for img in class_dir.glob("*.jpg"):
            rel = img.relative_to(subset_data_dir)
            records.append({"image_url": f"{base_uri}{rel.as_posix()}", "label": label})

with open(jsonl_path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

print("✅ JSONL created:", jsonl_path)

✅ JSONL created: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn/data/intel-image-classification/subset/annotations/train_annotations.jsonl


In [ ]:
import json
from pathlib import Path

# root of the dataset you're building locally
data_dir = subset_dir / "data"
jsonl_path = subset_dir / "train_annotations.jsonl"

records = []

for class_dir in data_dir.iterdir():
    if class_dir.is_dir():
        label = class_dir.name
        for img_path in class_dir.glob("*.*"):
            rel_path = img_path.relative_to(subset_dir)
            records.append({
                "image_url": rel_path.as_posix(),
                "label": label
            })

with open(jsonl_path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

print("✅ JSONL created:", jsonl_path)

In [46]:
%%writefile data/intel-image-classification/subset/annotations/MLTable

paths:
  - file: ./train_annotations.jsonl
transformations:
  - read_json_lines:
        encoding: utf8
        invalid_lines: error
        include_path_column: false
  - convert_column_types:
      - columns: image_url
        column_type: stream_info   

Writing data/intel-image-classification/subset/annotations/MLTable


In [48]:
subset_dir / "annotations"

PosixPath('/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn/data/intel-image-classification/subset/annotations')

In [50]:
import mltable

training_data  = Input(type=AssetTypes.MLTABLE, path=str(subset_dir / "annotations"))

tbl = mltable.load(uri=str(subset_dir / "annotations"))
df = tbl.to_pandas_dataframe()

In [53]:
df['image_url'][0]

StreamInfo[AmlDatastore](dmdp100/LocalUpload/dc3906e4087dd3e6d6fe82ebca3e1c35a9a6a6b2e06334350cbf8722526b90d5/data/buildings/10161.jpg)

In [54]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

mltable_asset = Data(
    name="intel-image-subset-mltable",
    path=str(subset_dir / "annotations"),
    type=AssetTypes.MLTABLE,
    datastore="dmdp100",
    description="Intel Image Classification subset prepared for AutoML"
)
registered_mltable = ml_client.data.create_or_update(mltable_asset)
print("✅ Registered MLTable asset:", registered_mltable.id)


Uploading annotations (0.34 MBs): 100%|██████████| 339182/339182 [00:00<00:00, 5373940.12it/s]




✅ Registered MLTable asset: /subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/data/intel-image-subset-mltable/versions/4


In [ ]:
ml_client.data._download

In [ ]:

# import json

# subset_dir = dataset_dir / "subset"
# jsonl_path = subset_dir / "train_annotations.jsonl"

# records = []

# for class_dir in subset_dir.iterdir():
#     if class_dir.is_dir():
#         label = class_dir.name
#         for img_path in class_dir.glob("*.jpg"):
#             record = {
#                 "image_url": str(img_path.resolve()),  # full path
#                 "label": label
#             }
#             records.append(record)

# # Write to JSONL
# with open(jsonl_path, "w", encoding="utf-8") as f:
#     for r in records:
#         f.write(json.dumps(r) + "\n")

# print(f"✅ JSONL created at: {jsonl_path}")
# print(f"Total records: {len(records)}")

In [33]:
data_asset = ml_client.data.get("intel-image-subset-folder", version="1")


In [34]:
data_asset.path

'azureml://subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/workspaces/polandaidevml-mlw/datastores/dmdp100/paths/LocalUpload/dc3906e4087dd3e6d6fe82ebca3e1c35a9a6a6b2e06334350cbf8722526b90d5/data/'

In [ ]:
import json

img_data_asset = ml_client.data.get("intel-image-subset-folder", version="1")
base_uri = img_data_asset.path

annotations_dir = subset_dir.parent / "annotations"
annotations_dir.mkdir(exist_ok=True)
jsonl_path = annotations_dir / "train_annotations.jsonl"
records = []

for class_dir in subset_dir.iterdir():
    if class_dir.is_dir():
        label = class_dir.name
        for img in class_dir.glob("*.jpg"):
            rel = img.relative_to(subset_dir)
            records.append({"image_url": f"{base_uri}{rel.as_posix()}", "label": label})

with open(jsonl_path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

print("✅ JSONL created:", jsonl_path)